# WM-811K Term Project — Step 3

## 카테고리 2: Knowledge Distillation Ablation

**구성**:
- **메인 KD**: ResNet-18 Teacher → Student CNN(0.39M)
- **Temperature ablation**: T = 1, 3, 5, 10, 20
- **Loss 비율 ablation**: alpha = 0.1, 0.5, 0.9 (soft label 비중)
- **Student 크기 ablation**: StudentCNN(0.39M) vs MobileNetV2(2.23M)
- **Teacher 크기 ablation**: ResNet-18(11M) vs ResNet-50(25M) — capacity gap 확인
- **Distillation 유무**: Step 2의 `student_none`(KD 없음)을 baseline으로 재활용

**사전 준비**: Step 1에서 학습한 Teacher 체크포인트 `teacher_none_aug0.pt`가 Drive에 있어야 함.
ResNet-50 Teacher는 이 노트북에서 새로 학습합니다.

**평가**: Accuracy + macro-F1 + 클래스별 F1 + 소수 클래스(Near-full/Donut/Random/Scratch) 평균 F1

## 1. 환경 설정

In [1]:
!nvidia-smi

Sun May 31 10:04:17 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   30C    P0             49W /  400W |       0MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [2]:
from google.colab import drive
drive.mount('/content/drive')
import os
PROJECT_DIR = '/content/drive/MyDrive/wm811k_term'
DATA_PATH = os.path.join(PROJECT_DIR, 'data', 'wm811k_64.npz')
CKPT_DIR = os.path.join(PROJECT_DIR, 'checkpoints')
RESULTS_DIR = os.path.join(PROJECT_DIR, 'results')
os.makedirs(CKPT_DIR, exist_ok=True); os.makedirs(RESULTS_DIR, exist_ok=True)
TEACHER_CKPT = os.path.join(CKPT_DIR, 'teacher_none_aug0.pt')
assert os.path.exists(DATA_PATH), DATA_PATH
assert os.path.exists(TEACHER_CKPT), f'Teacher 체크포인트 없음: {TEACHER_CKPT} (Step 1 먼저 실행)'
print('데이터/Teacher 체크포인트 확인 완료')

Mounted at /content/drive
데이터/Teacher 체크포인트 확인 완료


In [3]:
import random, numpy as np, torch
import torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
import torchvision.models as tvm
from sklearn.metrics import f1_score
import json

def set_seed(seed=42):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)
def get_device():
    return torch.device('cuda' if torch.cuda.is_available() else 'cpu')
set_seed(42); device = get_device()
print('Device:', device)

Device: cuda


In [4]:
d = np.load(DATA_PATH, allow_pickle=True)
X_train, y_train = d['X_train'], d['y_train']
X_test, y_test = d['X_test'], d['y_test']
CLASSES = list(d['classes']); NUM_CLASSES = len(CLASSES)
MINORITY = ['Near-full','Donut','Random','Scratch']
MIN_IDX = [CLASSES.index(c) for c in MINORITY]
print(f'Train {X_train.shape} Test {X_test.shape} Classes {NUM_CLASSES}')
print('소수 클래스:', MINORITY)

Train (138360, 64, 64) Test (34590, 64, 64) Classes 9
소수 클래스: ['Near-full', 'Donut', 'Random', 'Scratch']


## 2. 데이터 / 모델 / 평가 함수

In [5]:
class WaferDataset(Dataset):
    def __init__(self, X, y, train=False, augment=False):
        self.X, self.y, self.train, self.augment = X, y, train, augment
    def __len__(self): return len(self.X)
    def __getitem__(self, i):
        img = self.X[i].astype(np.float32)/2.0
        if self.train and self.augment:
            if random.random()<0.5: img = np.fliplr(img).copy()
            if random.random()<0.5: img = np.flipud(img).copy()
        return torch.from_numpy(img).unsqueeze(0), int(self.y[i])

def make_loaders(batch_size=128, augment=False):
    tr = DataLoader(WaferDataset(X_train,y_train,True,augment), batch_size=batch_size,
                    shuffle=True, num_workers=8, pin_memory=True)
    te = DataLoader(WaferDataset(X_test,y_test,False,False), batch_size=256,
                    shuffle=False, num_workers=8, pin_memory=True)
    return tr, te

def get_teacher_r18(n=NUM_CLASSES):
    m = tvm.resnet18(weights=None)
    m.conv1 = nn.Conv2d(1,64,3,stride=1,padding=1,bias=False); m.maxpool = nn.Identity()
    m.fc = nn.Linear(m.fc.in_features, n); return m

def get_teacher_r50(n=NUM_CLASSES):
    m = tvm.resnet50(weights=None)
    m.conv1 = nn.Conv2d(1,64,3,stride=1,padding=1,bias=False); m.maxpool = nn.Identity()
    m.fc = nn.Linear(m.fc.in_features, n); return m

class StudentCNN(nn.Module):
    def __init__(self, n=NUM_CLASSES):
        super().__init__()
        def b(ci,co): return nn.Sequential(nn.Conv2d(ci,co,3,padding=1,bias=False),
            nn.BatchNorm2d(co), nn.ReLU(inplace=True), nn.MaxPool2d(2))
        self.features = nn.Sequential(b(1,32),b(32,64),b(64,128),b(128,256))
        self.gap = nn.AdaptiveAvgPool2d(1); self.classifier = nn.Linear(256,n)
    def forward(self,x):
        x=self.features(x); x=self.gap(x).flatten(1); return self.classifier(x)

def get_student_cnn(n=NUM_CLASSES): return StudentCNN(n)

def get_student_mobilenet(n=NUM_CLASSES):
    m = tvm.mobilenet_v2(weights=None)
    old = m.features[0][0]
    m.features[0][0] = nn.Conv2d(1, old.out_channels, 3, stride=2, padding=1, bias=False)
    m.classifier[1] = nn.Linear(m.classifier[1].in_features, n); return m

def count_params(m): return sum(p.numel() for p in m.parameters() if p.requires_grad)

@torch.no_grad()
def evaluate(model, loader):
    model.eval(); preds=[]; trues=[]
    for x,y in loader:
        x=x.to(device, non_blocking=True)
        preds.append(model(x).argmax(1).cpu().numpy()); trues.append(y.numpy())
    pred,true=np.concatenate(preds),np.concatenate(trues)
    acc=(pred==true).mean()
    macro=f1_score(true,pred,average='macro',zero_division=0)
    per=f1_score(true,pred,average=None,labels=list(range(NUM_CLASSES)),zero_division=0)
    min_avg=float(np.array(per)[MIN_IDX].mean())
    return {'accuracy':float(acc),'macro_f1':float(macro),
            'minority_f1':min_avg,'per_class_f1':per.tolist()}

def print_metrics(m):
    print(f"  Accuracy: {m['accuracy']*100:.2f}%  Macro-F1: {m['macro_f1']:.4f}  소수평균-F1: {m['minority_f1']:.4f}")
    for c,f in zip(CLASSES, m['per_class_f1']): print(f'    {c:<12}{f:.4f}')

print('정의 완료. 모델 파라미터:')
for nm,fn in [('R18',get_teacher_r18),('R50',get_teacher_r50),('StudentCNN',get_student_cnn),('MobileNetV2',get_student_mobilenet)]:
    print(f'  {nm:<12}{count_params(fn()):>12,}')

정의 완료. 모델 파라미터:
  R18           11,172,297
  R50           23,517,641
  StudentCNN       390,633
  MobileNetV2    2,234,825


## 3. Teacher 로드 (ResNet-18) + 검증
Step 1에서 학습한 ResNet-18 Teacher를 로드합니다.

In [6]:
teacher_r18 = get_teacher_r18().to(device)
teacher_r18.load_state_dict(torch.load(TEACHER_CKPT, map_location=device))
teacher_r18.eval()
_, te_loader = make_loaders()
m = evaluate(teacher_r18, te_loader)
print('ResNet-18 Teacher 로드 완료')
print(f"  Accuracy {m['accuracy']*100:.2f}%  Macro-F1 {m['macro_f1']:.4f}  소수평균 {m['minority_f1']:.4f}")

ResNet-18 Teacher 로드 완료
  Accuracy 97.82%  Macro-F1 0.8885  소수평균 0.8708


## 4. KD 학습 함수
soft label(KL divergence, T² 스케일) + hard label(CE)를 alpha로 가중합.
- `T`: temperature
- `alpha`: soft label 비중 (alpha=0이면 순수 CE = distillation 없음)

In [7]:
def kd_loss(student_logits, teacher_logits, targets, T, alpha):
    soft = F.kl_div(F.log_softmax(student_logits/T, dim=1),
                    F.softmax(teacher_logits/T, dim=1),
                    reduction='batchmean') * (T*T)
    hard = F.cross_entropy(student_logits, targets)
    return alpha*soft + (1-alpha)*hard

def run_kd(teacher, student_fn, T=5.0, alpha=0.5, epochs=30, batch=128, lr=0.05, tag=None):
    set_seed(42)
    tag = tag or f'kd_T{T}_a{alpha}'
    tr_loader, te_loader = make_loaders(batch)
    student = student_fn().to(device)
    optimizer = torch.optim.SGD(student.parameters(), lr=lr, momentum=0.9, weight_decay=5e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    scaler = torch.amp.GradScaler('cuda', enabled=True)
    teacher.eval()
    ckpt_path = os.path.join(CKPT_DIR, f'{tag}.pt')
    print(f'>>> KD 시작: {tag} (T={T}, alpha={alpha}, epochs={epochs})')
    history, best=[], 0.0
    for ep in range(1, epochs+1):
        student.train(); loss_sum=tot=0
        for x,y in tr_loader:
            x,y=x.to(device,non_blocking=True),y.to(device,non_blocking=True)
            with torch.no_grad():
                t_logits = teacher(x)
            optimizer.zero_grad()
            with torch.amp.autocast('cuda', enabled=True):
                s_logits = student(x)
                loss = kd_loss(s_logits, t_logits, y, T, alpha)
            scaler.scale(loss).backward(); scaler.step(optimizer); scaler.update()
            loss_sum += loss.item()*x.size(0); tot += x.size(0)
        scheduler.step()
        mm = evaluate(student, te_loader)
        history.append({'epoch':ep,'loss':loss_sum/tot,**{k:mm[k] for k in ['accuracy','macro_f1','minority_f1']}})
        if ep%5==0 or ep==1:
            print(f"  Epoch {ep:02d} | loss {loss_sum/tot:.3f} | acc {mm['accuracy']:.4f} | macroF1 {mm['macro_f1']:.4f} | 소수 {mm['minority_f1']:.4f}")
        if mm['macro_f1']>best:
            best=mm['macro_f1']; torch.save(student.state_dict(), ckpt_path)
    student.load_state_dict(torch.load(ckpt_path, map_location=device))
    final = evaluate(student, te_loader)
    print(f"[{tag}] 최종: acc {final['accuracy']*100:.2f}% | macroF1 {final['macro_f1']:.4f} | 소수평균 {final['minority_f1']:.4f}")
    with open(os.path.join(RESULTS_DIR, f'{tag}.json'),'w') as f:
        json.dump({'T':T,'alpha':alpha,'tag':tag,**final,'classes':CLASSES,'history':history}, f, indent=2)
    return final

print('run_kd 정의 완료')

run_kd 정의 완료


## 5. Temperature Ablation (alpha=0.5 고정, ResNet-18 Teacher)
T = 1, 3, 5, 10, 20

In [8]:
temp_results = {}
for T in [1, 3, 5, 10, 20]:
    temp_results[T] = run_kd(teacher_r18, get_student_cnn, T=T, alpha=0.5,
                             tag=f'kd_r18_T{T}_a0.5')
print('\n=== Temperature Ablation 요약 ===')
print(f"{'T':<6}{'Acc':<10}{'Macro-F1':<12}{'소수평균':<10}")
for T in [1,3,5,10,20]:
    r=temp_results[T]
    print(f"{T:<6}{r['accuracy']*100:<10.2f}{r['macro_f1']:<12.4f}{r['minority_f1']:<10.4f}")

>>> KD 시작: kd_r18_T1_a0.5 (T=1, alpha=0.5, epochs=30)
  Epoch 01 | loss 0.235 | acc 0.9423 | macroF1 0.6086 | 소수 0.4397
  Epoch 05 | loss 0.112 | acc 0.9484 | macroF1 0.7572 | 소수 0.6950
  Epoch 10 | loss 0.091 | acc 0.9656 | macroF1 0.8013 | 소수 0.7471
  Epoch 15 | loss 0.071 | acc 0.9685 | macroF1 0.7986 | 소수 0.7167
  Epoch 20 | loss 0.043 | acc 0.9670 | macroF1 0.8125 | 소수 0.7722
  Epoch 25 | loss 0.011 | acc 0.9709 | macroF1 0.8443 | 소수 0.8111
  Epoch 30 | loss 0.006 | acc 0.9723 | macroF1 0.8491 | 소수 0.8147
[kd_r18_T1_a0.5] 최종: acc 97.24% | macroF1 0.8545 | 소수평균 0.8276
>>> KD 시작: kd_r18_T3_a0.5 (T=3, alpha=0.5, epochs=30)
  Epoch 01 | loss 0.753 | acc 0.9435 | macroF1 0.5333 | 소수 0.2817
  Epoch 05 | loss 0.321 | acc 0.9586 | macroF1 0.8166 | 소수 0.7913
  Epoch 10 | loss 0.254 | acc 0.9654 | macroF1 0.7916 | 소수 0.7037
  Epoch 15 | loss 0.201 | acc 0.9672 | macroF1 0.7409 | 소수 0.5992
  Epoch 20 | loss 0.135 | acc 0.9693 | macroF1 0.8353 | 소수 0.8108
  Epoch 25 | loss 0.071 | acc 0.9741 

## 6. Loss 비율(alpha) Ablation (T=5 고정)
alpha = 0.1, 0.5, 0.9. (alpha=0은 Step 2의 student_none과 동일 = distillation 없음)

In [9]:
alpha_results = {}
for a in [0.1, 0.5, 0.9]:
    alpha_results[a] = run_kd(teacher_r18, get_student_cnn, T=5.0, alpha=a,
                              tag=f'kd_r18_T5_a{a}')
print('\n=== Alpha Ablation 요약 ===')
print(f"{'alpha':<8}{'Acc':<10}{'Macro-F1':<12}{'소수평균':<10}")
for a in [0.1,0.5,0.9]:
    r=alpha_results[a]
    print(f"{a:<8}{r['accuracy']*100:<10.2f}{r['macro_f1']:<12.4f}{r['minority_f1']:<10.4f}")

>>> KD 시작: kd_r18_T5_a0.1 (T=5.0, alpha=0.1, epochs=30)
  Epoch 01 | loss 0.413 | acc 0.9490 | macroF1 0.5521 | 소수 0.2747
  Epoch 05 | loss 0.191 | acc 0.9404 | macroF1 0.7348 | 소수 0.6394
  Epoch 10 | loss 0.152 | acc 0.9712 | macroF1 0.8377 | 소수 0.7903
  Epoch 15 | loss 0.121 | acc 0.9704 | macroF1 0.7864 | 소수 0.6815
  Epoch 20 | loss 0.078 | acc 0.9686 | macroF1 0.8187 | 소수 0.7832
  Epoch 25 | loss 0.039 | acc 0.9738 | macroF1 0.8581 | 소수 0.8249
  Epoch 30 | loss 0.031 | acc 0.9737 | macroF1 0.8568 | 소수 0.8254
[kd_r18_T5_a0.1] 최종: acc 97.40% | macroF1 0.8612 | 소수평균 0.8341
>>> KD 시작: kd_r18_T5_a0.5 (T=5.0, alpha=0.5, epochs=30)
  Epoch 01 | loss 0.827 | acc 0.9258 | macroF1 0.4702 | 소수 0.2328
  Epoch 05 | loss 0.357 | acc 0.9515 | macroF1 0.7801 | 소수 0.7227
  Epoch 10 | loss 0.287 | acc 0.9663 | macroF1 0.8023 | 소수 0.7358
  Epoch 15 | loss 0.230 | acc 0.9592 | macroF1 0.8034 | 소수 0.7461
  Epoch 20 | loss 0.168 | acc 0.9681 | macroF1 0.8126 | 소수 0.7675
  Epoch 25 | loss 0.101 | acc 0.9

## 7. Student 크기 Ablation (T=5, alpha=0.5)
StudentCNN(0.39M) vs MobileNetV2(2.23M)

In [10]:
size_results = {}
size_results['StudentCNN_0.39M'] = run_kd(teacher_r18, get_student_cnn, T=5.0, alpha=0.5, tag='kd_r18_studentcnn')
size_results['MobileNetV2_2.23M'] = run_kd(teacher_r18, get_student_mobilenet, T=5.0, alpha=0.5, tag='kd_r18_mobilenet')
print('\n=== Student 크기 Ablation 요약 ===')
print(f"{'Student':<20}{'Acc':<10}{'Macro-F1':<12}{'소수평균':<10}")
for k,r in size_results.items():
    print(f"{k:<20}{r['accuracy']*100:<10.2f}{r['macro_f1']:<12.4f}{r['minority_f1']:<10.4f}")

>>> KD 시작: kd_r18_studentcnn (T=5.0, alpha=0.5, epochs=30)
  Epoch 01 | loss 0.827 | acc 0.9258 | macroF1 0.4702 | 소수 0.2328
  Epoch 05 | loss 0.357 | acc 0.9515 | macroF1 0.7801 | 소수 0.7227
  Epoch 10 | loss 0.287 | acc 0.9663 | macroF1 0.8023 | 소수 0.7358
  Epoch 15 | loss 0.230 | acc 0.9592 | macroF1 0.8034 | 소수 0.7461
  Epoch 20 | loss 0.168 | acc 0.9681 | macroF1 0.8126 | 소수 0.7675
  Epoch 25 | loss 0.101 | acc 0.9748 | macroF1 0.8632 | 소수 0.8302
  Epoch 30 | loss 0.075 | acc 0.9755 | macroF1 0.8692 | 소수 0.8423
[kd_r18_studentcnn] 최종: acc 97.56% | macroF1 0.8732 | 소수평균 0.8498
>>> KD 시작: kd_r18_mobilenet (T=5.0, alpha=0.5, epochs=30)
  Epoch 01 | loss 0.671 | acc 0.9338 | macroF1 0.5193 | 소수 0.3264
  Epoch 05 | loss 0.377 | acc 0.9569 | macroF1 0.7497 | 소수 0.6464
  Epoch 10 | loss 0.331 | acc 0.9640 | macroF1 0.7768 | 소수 0.6686
  Epoch 15 | loss 0.282 | acc 0.9545 | macroF1 0.7496 | 소수 0.6424
  Epoch 20 | loss 0.223 | acc 0.9731 | macroF1 0.8407 | 소수 0.7842
  Epoch 25 | loss 0.142 |

## 8. Teacher 크기 Ablation — ResNet-50 학습 후 KD 비교 (capacity gap)
먼저 ResNet-50 Teacher를 학습하고, 같은 설정(T=5, alpha=0.5)으로 Student를 KD하여
ResNet-18 Teacher 결과와 비교합니다. Teacher가 강할수록 좋은지(capacity gap) 확인.

In [12]:
# 8-1. ResNet-50 Teacher 학습 (augment=False, 30 epoch)
def train_teacher_r50(epochs=30, batch=128, lr=0.1):
    set_seed(42)
    tr_loader, te_loader = make_loaders(batch)
    model = get_teacher_r50().to(device)
    opt = torch.optim.SGD(model.parameters(), lr=lr, momentum=0.9, weight_decay=5e-4)
    sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    scaler = torch.amp.GradScaler('cuda', enabled=True)
    ckpt = os.path.join(CKPT_DIR, 'teacher_r50.pt'); best=0.0
    print('>>> ResNet-50 Teacher 학습')
    for ep in range(1, epochs+1):
        model.train()
        for x,y in tr_loader:
            x,y=x.to(device,non_blocking=True),y.to(device,non_blocking=True)
            opt.zero_grad()
            with torch.amp.autocast('cuda', enabled=True):
                loss = F.cross_entropy(model(x), y)
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
        sch.step()
        mm = evaluate(model, te_loader)
        if ep%5==0 or ep==1:
            print(f"  Epoch {ep:02d} | acc {mm['accuracy']:.4f} | macroF1 {mm['macro_f1']:.4f}")
        if mm['macro_f1']>best:
            best=mm['macro_f1']; torch.save(model.state_dict(), ckpt)
    model.load_state_dict(torch.load(ckpt, map_location=device))
    final = evaluate(model, te_loader)
    print(f"ResNet-50 Teacher 최종: acc {final['accuracy']*100:.2f}% | macroF1 {final['macro_f1']:.4f}")
    return model

teacher_r50 = train_teacher_r50()

>>> ResNet-50 Teacher 학습
  Epoch 01 | acc 0.9151 | macroF1 0.3282
  Epoch 05 | acc 0.9446 | macroF1 0.6595
  Epoch 10 | acc 0.9280 | macroF1 0.6268
  Epoch 15 | acc 0.9578 | macroF1 0.7353
  Epoch 20 | acc 0.9688 | macroF1 0.8025
  Epoch 25 | acc 0.9756 | macroF1 0.8658
  Epoch 30 | acc 0.9776 | macroF1 0.8752
ResNet-50 Teacher 최종: acc 97.74% | macroF1 0.8756


In [13]:
# 8-2. ResNet-50 Teacher로 Student KD (동일 설정 T=5, alpha=0.5)
kd_r50 = run_kd(teacher_r50, get_student_cnn, T=5.0, alpha=0.5, tag='kd_r50_studentcnn')

print('\n=== Teacher 크기 Ablation 요약 (Student=StudentCNN, T=5, alpha=0.5) ===')
print(f"{'Teacher':<16}{'Acc':<10}{'Macro-F1':<12}{'소수평균':<10}")
r18 = size_results['StudentCNN_0.39M']
print(f"{'ResNet-18':<16}{r18['accuracy']*100:<10.2f}{r18['macro_f1']:<12.4f}{r18['minority_f1']:<10.4f}")
print(f"{'ResNet-50':<16}{kd_r50['accuracy']*100:<10.2f}{kd_r50['macro_f1']:<12.4f}{kd_r50['minority_f1']:<10.4f}")

>>> KD 시작: kd_r50_studentcnn (T=5.0, alpha=0.5, epochs=30)
  Epoch 01 | loss 0.669 | acc 0.9475 | macroF1 0.5969 | 소수 0.3801
  Epoch 05 | loss 0.247 | acc 0.9680 | macroF1 0.8323 | 소수 0.7929
  Epoch 10 | loss 0.197 | acc 0.9680 | macroF1 0.8372 | 소수 0.7982
  Epoch 15 | loss 0.161 | acc 0.9718 | macroF1 0.8506 | 소수 0.8280
  Epoch 20 | loss 0.123 | acc 0.9702 | macroF1 0.8367 | 소수 0.8122
  Epoch 25 | loss 0.083 | acc 0.9753 | macroF1 0.8715 | 소수 0.8460
  Epoch 30 | loss 0.066 | acc 0.9760 | macroF1 0.8746 | 소수 0.8496
[kd_r50_studentcnn] 최종: acc 97.60% | macroF1 0.8746 | 소수평균 0.8496

=== Teacher 크기 Ablation 요약 (Student=StudentCNN, T=5, alpha=0.5) ===
Teacher         Acc       Macro-F1    소수평균      
ResNet-18       97.56     0.8732      0.8498    
ResNet-50       97.60     0.8746      0.8496    


---
### Step 3 완료 후
- **Distillation 유무**: Step 2 `student_none`(KD 없음, macro-F1 0.8606) vs KD 적용 결과 비교
- **Temperature/alpha**: 최적 조합 확인
- **Student 크기**: 0.39M vs 2.23M에서 KD 효과 차이
- **Teacher 크기**: ResNet-18 vs ResNet-50 → capacity gap 존재 여부
- 다음(Step 4)에서 **불균형 처리(focal) + KD 시너지** 2x2 분석